In [1]:
import numpy as np
from joblib import dump
from sklearn.ensemble import RandomForestClassifier

train = np.loadtxt("Datasets/pendigits.tra", delimiter=",")
test = np.loadtxt("Datasets/pendigits.tes", delimiter=",")
Xtr, ytr = train[:, :16] / 100, train[:, 16].astype(int)
Xte, yte = test[:, :16] / 100, test[:, 16].astype(int)

model = RandomForestClassifier(n_estimators=80, random_state=42)
model.fit(Xtr, ytr)
print("Test Accuracy:", round(model.score(Xte, yte), 4))

dump(model, "digit_model.joblib", compress=3)
print("Model saved as digit_model.joblib")


Test Accuracy: 0.9634
Model saved as digit_model.joblib


In [2]:
%%writefile app.py
from fastapi import FastAPI
from joblib import load
from pydantic import BaseModel

app = FastAPI()
model = load("digit_model.joblib")


class Trace(BaseModel):
    coordinates: list[int]


@app.post("/predict")
def predict(trace: Trace):
    x = [v / 100 for v in trace.coordinates]
    digit = model.predict([x])[0]
    return {"digit": int(digit)}


Writing app.py


In [3]:
import sys
import time
import subprocess
import requests

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", "8000"])

sample = {"coordinates": test[0, :16].astype(int).tolist()}
print("Request:", sample)

url = "http://127.0.0.1:8000/predict"
for i in range(30):
    try:
        r = requests.post(url, json=sample)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

print("Status code:", r.status_code)
print("Response:", r.json())
print("Actual digit:", yte[0])

server.terminate()


Request: {'coordinates': [88, 92, 2, 99, 16, 66, 94, 37, 70, 0, 0, 24, 42, 65, 100, 100]}
Status code: 200
Response: {'digit': 8}
Actual digit: 8
